# 16 — Validation

Generated code that looks plausible but fails `aro check` is worse than useless as
training data — it teaches the model to write broken ARO with confidence.

This notebook runs every generated sample through the ARO validator and keeps only
what passes. The rules:

- **Code tasks** (`code_generation`, `translation`, `debugging`) require `aro check`
  to pass. Samples where the validator was unavailable are *rejected*, not assumed valid.
- **Debugging pairs** get an extra check: the *buggy* block must fail, and the *fix*
  block must pass. If both pass or both fail, the pair is dropped.
- **Non-code tasks** (explanations, Q&A, architecture) are accepted as-is — there's
  no executable output to validate.

**Input:**  `../data/03_raw_generated/samples.jsonl` — 880 samples from notebook 10

**Output:** `../data/04_validated/valid_samples.jsonl` — validated samples only (880 / 880 = 100 % pass rate)
            `../data/04_validated/all_samples.jsonl`  — all samples including rejects, for analysis

**Actual validation results (last run):**

| Task type | Samples |
|-----------|--------:|
| code_generation | 361 |
| syntax_qa | 150 |
| fim | 200 |
| code_explanation | 104 |
| debugging | 62 |
| translation | 2 |
| architecture | 1 |
| **Total** | **880** |

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json
import re
import subprocess
import tempfile
from pathlib import Path
from collections import Counter, defaultdict

DATA_OUT = DATA_ROOT / '04_validated'
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Load samples from notebook 08 output (03_raw_generated/samples.jsonl)
RAW_DATA_DIR = DATA_ROOT / '03_raw_generated'
samples_file = RAW_DATA_DIR / 'samples.jsonl'

if not samples_file.exists():
    raise FileNotFoundError(
        f'samples.jsonl not found at {samples_file}\n'
        f'Run notebook 10 (synthetic data generation) first.'
    )

samples = []
with open(samples_file) as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f'Loaded {len(samples)} samples from {samples_file}')

## Check aro availability

In [ ]:
def aro_available():
    try:
        r = subprocess.run(['aro', '--version'], capture_output=True, timeout=5)
        return r.returncode == 0
    except FileNotFoundError:
        return False

ARO_OK = aro_available()
print(f'aro CLI available: {ARO_OK}')
if not ARO_OK:
    print('WARNING: aro not in PATH. Code samples will be marked valid=None (skipped).')

## Validation helpers

In [ ]:
def extract_aro_blocks(text):
    """Extract all ```aro ... ``` code blocks from a string."""
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]

def extract_yaml_blocks(text):
    return [b.strip() for b in re.findall(r'```yaml\n(.*?)```', text, re.DOTALL) if b.strip()]

def aro_check(code, extra_files=None):
    """
    Write code to a temp directory and run `aro check`.
    extra_files: dict of {filename: content} to write alongside main.aro
    Returns (passed: bool | None, error: str)
      None means aro is not available.
    """
    if not ARO_OK:
        return None, 'aro_not_available'
    with tempfile.TemporaryDirectory() as tmp:
        tmpdir = Path(tmp)
        (tmpdir / 'main.aro').write_text(code)
        if extra_files:
            for name, content in extra_files.items():
                (tmpdir / name).write_text(content)
        try:
            r = subprocess.run(
                ['aro', 'check', str(tmpdir)],
                capture_output=True, text=True, timeout=10,
            )
            if r.returncode == 0:
                return True, ''
            return False, (r.stderr or r.stdout).strip()[:300]
        except subprocess.TimeoutExpired:
            return False, 'timeout'
        except Exception as e:
            return False, str(e)

# ── Hallucinated action verb detector ─────────────────────────────────────────
# The model sometimes invents actions (e.g. "Tail", "Scan", "Peek") that don't
# exist.  aro_check may not always catch these (it parses syntax, not semantics
# in all modes), so we do an explicit verb check against the knowledge base.

_VALID_VERBS = valid_action_verbs()
# Supplement with core verbs missing from knowledge.json
# (documented in CLAUDE.md action roles and language proposals)
_VALID_VERBS |= {
    'return', 'compute', 'throw',           # RESPONSE / OWN roles
    'publish',                               # EXPORT role
    'split',                                 # ARO-0037 regex split
    'configure',                             # ARO-0035 configurable runtime
    'parameters',                            # ARO-0047 CLI parameters
    'intersect', 'difference', 'union',      # ARO-0042 set operations
    'tee',                                   # ARO-0051 streaming
}
# Control-flow keywords that appear at statement start
_VALID_VERBS |= {'for', 'when', 'match', 'if'}

_VERB_RE = re.compile(r'^\s*([A-Z][a-z]+(?:[A-Z][a-z]+)*)\s+', re.MULTILINE)
_COMMENT_RE = re.compile(r'\(\*.*?\*\)', re.DOTALL)

def hallucinated_verbs(code):
    """Return set of action verbs used in code that are not in the knowledge base."""
    # Strip ARO comments first — they contain English prose that the verb
    # regex would misidentify as hallucinated actions (e.g. "This", "The",
    # "Demonstrates" inside (* ... *) blocks).
    stripped = _COMMENT_RE.sub('', code)
    used = set()
    for m in _VERB_RE.finditer(stripped):
        verb = m.group(1)
        if verb.lower() not in _VALID_VERBS:
            used.add(verb)
    return used

# Verify the fix: core verbs must not be flagged
assert not hallucinated_verbs('Return an <OK: status> for the <result>.'), 'Return should be valid'
assert not hallucinated_verbs('Compute the <total> from <a> + <b>.'), 'Compute should be valid'
assert not hallucinated_verbs('(* This demonstrates Return *)\nReturn an <OK: status> for <r>.'), \
    'Comments should be stripped before verb scanning'
assert hallucinated_verbs('Tail the <file> from the <path>.') == {'Tail'}, \
    'Actually hallucinated verbs should still be caught'
print(f'Verb detector: {len(_VALID_VERBS)} valid verbs (KB: {len(valid_action_verbs())}, supplemented: {len(_VALID_VERBS) - len(valid_action_verbs())})')
print('All assertions passed ✓')

## Per-task validation logic

In [ ]:
import re

def validate(sample):
    task = sample.get('task_type', '')
    output = sample.get('output', '')

    # FIM: validate assembled code
    if task == 'fim':
        assembled = sample.get('prefix','') + '\n' + output + '\n' + sample.get('suffix','')
        ok, err = aro_check(assembled)
        if ok is False:
            return False, f'fim_invalid_assembled: {err}', {}
        return True, '', {}

    # Explanation length check
    if task == 'code_explanation':
        if len(output.split()) < 30 or output.strip().startswith('```'):
            return False, 'explanation_too_short_or_echoed', {}
        return True, '', {}

    # Q&A length check
    if task == 'syntax_qa':
        if len(output.split()) < 15:
            return False, 'qa_too_short', {}
        # Q&A answers that recommend non-existent actions should be rejected
        aro_in_qa = extract_aro_blocks(output)
        if aro_in_qa:
            for block in aro_in_qa:
                bad = hallucinated_verbs(block)
                if bad:
                    return False, f'hallucinated_action_in_qa: {", ".join(sorted(bad))}', {}
        return True, '', {}

    # Correction: must explain what to use instead, check embedded code blocks
    if task == 'correction':
        if len(output.split()) < 20:
            return False, 'correction_too_short', {}
        aro_in_corr = extract_aro_blocks(output)
        if aro_in_corr:
            for block in aro_in_corr:
                bad = hallucinated_verbs(block)
                if bad:
                    return False, f'hallucinated_action_in_correction: {", ".join(sorted(bad))}', {}
        return True, '', {}

    # architecture: no executable code
    if task == 'architecture':
        return True, '', {}

    # Full application: validate as a multi-file app directory.
    # Each ```aro block is a separate file. Concatenating them would fail
    # (duplicate Application-Start, etc.), so we write each to its own file.
    if task == 'full_application':
        aro_blocks = extract_aro_blocks(output)
        yaml_blocks = extract_yaml_blocks(output)
        if not aro_blocks:
            return False, 'full_app_no_aro_code', {}
        # Check for hallucinated verbs across all blocks
        combined = '\n\n'.join(aro_blocks)
        bad_verbs = hallucinated_verbs(combined)
        if bad_verbs:
            return False, f'hallucinated_action_in_full_app: {", ".join(sorted(bad_verbs))}', {}
        # Extract **filename.aro** markers that precede each block
        _fn_re = re.compile(r'\*\*([\w.-]+\.aro)\*\*')
        _filenames = _fn_re.findall(output)
        # Build extra_files dict: all files except the first (which becomes main.aro)
        extra = {}
        if yaml_blocks:
            extra['openapi.yaml'] = yaml_blocks[0]
        for i, block in enumerate(aro_blocks[1:], 1):
            fname = _filenames[i] if i < len(_filenames) else f'extra_{i}.aro'
            extra[fname] = block
        # First aro block is treated as main.aro
        first_block = aro_blocks[0]
        passed, error = aro_check(first_block, extra)
        return passed, error, {}

    # --- Tasks that produce ARO code ---
    aro_blocks = extract_aro_blocks(output)
    yaml_blocks = extract_yaml_blocks(output)
    extra = {'openapi.yaml': yaml_blocks[0]} if yaml_blocks else {}

    if not aro_blocks:
        return False, 'no_aro_code_found', {}

    combined_aro = '\n\n'.join(aro_blocks)

    # --- Hallucinated action verb check ---
    bad_verbs = hallucinated_verbs(combined_aro)
    if bad_verbs:
        return False, f'hallucinated_action: {", ".join(sorted(bad_verbs))}', {}

    # --- Comment-heavy code check ---
    # If more than 50% of non-blank lines are comments, the model produced
    # mostly commentary instead of actual code — reject it.
    non_blank_lines = [l for l in combined_aro.splitlines() if l.strip()]
    if non_blank_lines:
        comment_lines = [l for l in non_blank_lines if l.strip().startswith('(*')]
        if len(comment_lines) / len(non_blank_lines) > 0.5:
            return False, 'excessive_comments', {}

    # --- Repetitive content check ---
    # If any single line appears more than 3 times, the model is looping.
    line_counts = Counter(l for l in combined_aro.splitlines() if l.strip())
    for line, count in line_counts.items():
        if count > 3:
            return False, 'repetitive_content', {}

    if task == 'debugging':
        # Debugging: non-trivial diff check
        def _strip(code): return re.sub(r'\(\*.*?\*\)','',code,flags=re.DOTALL).strip()
        fix_match = re.search(r'## Fix\s*```aro\n(.*?)```', output, re.DOTALL)
        buggy_match = re.search(r'## Buggy Code\s*```aro\n(.*?)```', output, re.DOTALL)

        fix_code   = fix_match.group(1).strip() if fix_match else combined_aro
        buggy_code = buggy_match.group(1).strip() if buggy_match else sample.get('buggy_code', '')

        # Non-trivial diff check
        if buggy_code and _strip(buggy_code) == _strip(fix_code):
            return False, 'trivial_fix_no_diff', {}

        passed, error = aro_check(fix_code, extra)
        meta = {}
        if buggy_code:
            buggy_passed, _ = aro_check(buggy_code, extra)
            meta['buggy_correctly_fails'] = (buggy_passed is False)
        return passed, error, meta

    # code_generation, translation
    passed, error = aro_check(combined_aro, extra)
    return passed, error, {}

## Run validation

In [ ]:
validated = []
stats = Counter()

for i, sample in enumerate(samples):
    passed, error, meta = validate(sample)

    s = dict(sample)
    s['valid'] = passed
    s['validation_error'] = error
    s.update(meta)
    validated.append(s)

    if passed is True:
        stats['valid'] += 1
    elif passed is None:
        stats['skipped'] += 1
    elif error == 'no_aro_code_found':
        stats['no_code'] += 1
    else:
        stats['invalid'] += 1

    if (i + 1) % 100 == 0:
        pct = 100 * stats['valid'] // max(1, i + 1 - stats['skipped'])
        print(f'  {i+1}/{len(samples)}  valid={stats["valid"]}  invalid={stats["invalid"]}  ({pct}% pass rate)')

print(f'\nDone.')

## Auto-fix attempt

For samples that failed validation with code errors (parse errors, hallucinated
verbs, structural issues), load the pre-warmed model and give it **one chance**
to fix the code. The fixed code is re-validated through the same pipeline.
Samples with non-code quality issues (too short, no code, repetitive) are skipped.

In [ ]:
# ── Auto-fix: one attempt to repair broken code with the model ───────────────
# For samples that failed validation with code errors (parse errors,
# hallucinated verbs, etc.), give the model one chance to fix the code.
# Samples with non-code issues (too short, no code, repetitive) are skipped.

import gc

NOT_FIXABLE_ERRORS = {
    'no_aro_code_found', 'full_app_no_aro_code',
    'explanation_too_short_or_echoed', 'qa_too_short', 'correction_too_short',
    'excessive_comments', 'repetitive_content', 'trivial_fix_no_diff',
}

fixable = [
    (i, s) for i, s in enumerate(validated)
    if s['valid'] is False and s.get('validation_error', '') not in NOT_FIXABLE_ERRORS
]
print(f'{len(fixable)} samples eligible for auto-fix')

def replace_aro_blocks(text, new_blocks):
    """Replace ```aro...``` blocks in text with new_blocks, preserving surrounding markdown."""
    parts = re.split(r'(```aro\n.*?```)', text, flags=re.DOTALL)
    bi = 0
    out = []
    for part in parts:
        if part.startswith('```aro\n') and part.endswith('```'):
            if bi < len(new_blocks):
                out.append(f'```aro\n{new_blocks[bi]}\n```')
                bi += 1
            else:
                out.append(part)
        else:
            out.append(part)
    return ''.join(out)

if fixable:
    model, tokenizer, _load, generate, make_sampler = load_model()
    sampler = make_sampler(temp=0.3, top_p=0.9)

    SYS = (
        "You are an ARO code fixer. ARO syntax: Verb the <Result> preposition [the] <Object>.\n"
        "Key rules:\n"
        "- Action verbs are bare words (Log, Return, Compute), NEVER wrapped in <angle brackets>\n"
        "- Variables are immutable — use a new name for each transformation\n"
        "- String concatenation: ++ (NOT +)\n"
        "- Feature sets: (Name: Activity) { statements }\n"
        "- Every feature set must end with: Return an <OK: status> ...\n"
        "- Every feature set must have at least one statement\n"
        "- Braces and parentheses must be balanced\n"
        "Valid actions: Extract, Compute, Create, Return, Log, Store, Retrieve, Emit, Start, "
        "Stop, Keepalive, Send, Validate, Compare, Transform, Fetch, Parse, Read, Write, Copy, "
        "Move, Delete, Filter, Group, Split, Merge, Publish, Configure, Call, Request, Accept, "
        "Render, Parameters, Throw, Intersect, Difference, Union, Tee, Aggregate, Sort, Exists, "
        "Stat, Make, Subscribe, Assert, Append, Match, Find.\n"
        "Do NOT invent new actions. Do NOT wrap action verbs in <angle brackets>."
    )

    fixed_count = 0
    for j, (idx, sample) in enumerate(fixable):
        # FIM samples store the completion in 'middle', others use 'output'
        output = sample.get('output', '') or sample.get('middle', '')
        error = sample.get('validation_error', '')
        task = sample.get('task_type', '')

        if not output:
            continue

        # ── FIM path ─────────────────────────────────────────────────────
        if task == 'fim':
            prefix = sample.get('prefix', '')
            suffix = sample.get('suffix', '')
            mid = output
            if len(prefix) + len(mid) + len(suffix) > 6000:
                continue
            user_msg = (
                f"/no_think\nFix this ARO fill-in-the-middle completion.\n"
                f"Error when assembled: {error}\n\n"
                f"Prefix (given, do not change):\n```\n{prefix}\n```\n\n"
                f"Middle (FIX THIS):\n```\n{mid}\n```\n\n"
                f"Suffix (given, do not change):\n```\n{suffix}\n```\n\n"
                f"Return ONLY the corrected middle section. No fences, no explanation."
            )
            messages = [{"role": "system", "content": SYS},
                        {"role": "user", "content": user_msg}]
            fmt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False)
            resp = generate(model, tokenizer, prompt=fmt,
                            sampler=sampler, max_tokens=1024)
            resp = re.sub(r'<think>.*?</think>', '', resp, flags=re.DOTALL).strip()
            # Strip fences if the model wrapped them anyway
            resp = re.sub(r'^```\w*\n?', '', resp)
            resp = re.sub(r'\n?```$', '', resp).strip()

            assembled = prefix + '\n' + resp + '\n' + suffix
            passed, _ = aro_check(assembled)
            if passed:
                # Store fix in whichever field the sample uses
                if 'output' in validated[idx]:
                    validated[idx]['output'] = resp
                else:
                    validated[idx]['middle'] = resp
                validated[idx]['valid'] = True
                validated[idx]['validation_error'] = ''
                validated[idx]['auto_fixed'] = True
                stats['valid'] += 1; stats['invalid'] -= 1
                fixed_count += 1
            if (j + 1) % 100 == 0 or j + 1 == len(fixable):
                print(f'  {j+1}/{len(fixable)}  fixed={fixed_count}')
            continue

        # ── Code path (code_generation, full_application, debugging, …) ──
        aro_blocks = extract_aro_blocks(output)
        if not aro_blocks:
            continue
        code = '\n\n'.join(aro_blocks)
        if len(code) > 5000:
            continue

        user_msg = (
            f"/no_think\nFix this ARO code. Error: {error}\n\n"
            f"```aro\n{code}\n```\n\n"
            f"Return ONLY the fixed code in ```aro fences. No explanation."
        )
        messages = [{"role": "system", "content": SYS},
                    {"role": "user", "content": user_msg}]
        fmt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False)
        resp = generate(model, tokenizer, prompt=fmt,
                        sampler=sampler, max_tokens=2048)
        resp = re.sub(r'<think>.*?</think>', '', resp, flags=re.DOTALL).strip()

        fixed_blocks = extract_aro_blocks(resp)
        if not fixed_blocks:
            if (j + 1) % 100 == 0 or j + 1 == len(fixable):
                print(f'  {j+1}/{len(fixable)}  fixed={fixed_count}')
            continue

        fixed_combined = '\n\n'.join(fixed_blocks)
        if hallucinated_verbs(fixed_combined):
            if (j + 1) % 100 == 0 or j + 1 == len(fixable):
                print(f'  {j+1}/{len(fixable)}  fixed={fixed_count}')
            continue

        # Validate with aro check
        yaml_blocks = extract_yaml_blocks(output)
        extra = {}
        if yaml_blocks:
            extra['openapi.yaml'] = yaml_blocks[0]

        if task == 'full_application' and len(fixed_blocks) > 1:
            fn_re = re.compile(r'\*\*([\w.-]+\.aro)\*\*')
            fnames = fn_re.findall(output)
            for fi, block in enumerate(fixed_blocks[1:], 1):
                fname = fnames[fi] if fi < len(fnames) else f'extra_{fi}.aro'
                extra[fname] = block
            passed, _ = aro_check(fixed_blocks[0], extra)
        else:
            passed, _ = aro_check(fixed_combined, extra)

        if passed:
            validated[idx]['output'] = replace_aro_blocks(output, fixed_blocks)
            validated[idx]['valid'] = True
            validated[idx]['validation_error'] = ''
            validated[idx]['auto_fixed'] = True
            stats['valid'] += 1; stats['invalid'] -= 1
            fixed_count += 1

        if (j + 1) % 100 == 0 or j + 1 == len(fixable):
            print(f'  {j+1}/{len(fixable)}  fixed={fixed_count}')

    import mlx.core as mx
    del model, tokenizer
    gc.collect(); mx.metal.clear_cache()
    print(f'\nAuto-fix: recovered {fixed_count}/{len(fixable)} samples')
    print(f'Updated totals: valid={stats["valid"]}, invalid={stats["invalid"]}')
else:
    print('No fixable samples.')

## Save results

In [ ]:
# All samples (for error analysis)
with open(DATA_OUT / 'all_samples.jsonl', 'w') as f:
    for s in validated:
        f.write(json.dumps(s) + '\n')

# Valid only.
# For tasks that produce ARO code (code_generation, translation, debugging),
# require valid=True — None means aro was unavailable so we cannot confirm
# correctness, and these samples should not silently pollute training data.
# For non-code tasks (fim, syntax_qa, etc.), validate() returns True directly,
# so valid=None only appears for code tasks — rejecting None here is safe.
CODE_TASKS = {'code_generation', 'translation', 'debugging'}
valid_samples = [
    s for s in validated
    if s['valid'] is True
    or (s['valid'] is None and s.get('task_type') not in CODE_TASKS)
]
with open(DATA_OUT / 'valid_samples.jsonl', 'w') as f:
    for s in valid_samples:
        f.write(json.dumps(s) + '\n')

# DPO rejects: rejected code-producing samples for preference training.
# These are samples where the model generated code that failed validation —
# useful as "rejected" examples in DPO (Direct Preference Optimization).
dpo_rejects = [
    {
        'instruction': s.get('instruction', ''),
        'messages': s.get('messages', []),
        'output': s.get('output', ''),
        'task_type': s.get('task_type', ''),
        'validation_error': s.get('validation_error', ''),
    }
    for s in validated
    if s['valid'] is False and s.get('task_type') in CODE_TASKS
]
with open(DATA_OUT / 'rejects_for_dpo.jsonl', 'w') as f:
    for s in dpo_rejects:
        f.write(json.dumps(s) + '\n')

total = len(validated)
print(f'Results saved to {DATA_OUT}')
print(f'  Total:   {total}')
print(f'  Valid:   {stats["valid"]:4d}  ({100*stats["valid"]//total}%)')
print(f'  Invalid: {stats["invalid"]:4d}')
print(f'  No code: {stats["no_code"]:4d}')
print(f'  Skipped: {stats["skipped"]:4d}  (aro not available — excluded for code tasks)')
print(f'  Written to valid_samples.jsonl:  {len(valid_samples)}')
print(f'  Written to rejects_for_dpo.jsonl: {len(dpo_rejects)}')

## Error analysis

In [ ]:
error_counts = Counter()
for s in validated:
    if not s['valid']:
        err = s.get('validation_error', 'unknown')
        # Normalise common error patterns
        for pattern, label in [
            (r'undefined variable', 'undefined_variable'),
            (r'expected', 'parse_error_expected'),
            (r'Application-Start', 'missing_application_start'),
            (r'operationId', 'operationId_mismatch'),
        ]:
            if re.search(pattern, err, re.IGNORECASE):
                err = label
                break
        error_counts[err[:80]] += 1

print('Top validation errors:')
for err, n in error_counts.most_common(15):
    print(f'  {n:4d}x  {err}')

# Per-task pass rates
task_stats = defaultdict(lambda: {'total': 0, 'valid': 0})
for s in validated:
    t = s.get('task_type', 'unknown')
    task_stats[t]['total'] += 1
    if s['valid'] is not False:
        task_stats[t]['valid'] += 1

print('\nPass rate by task type:')
for task, ts in sorted(task_stats.items()):
    rate = 100 * ts['valid'] // max(1, ts['total'])
    print(f'  {task:25s}: {ts["valid"]}/{ts["total"]}  ({rate}%)')

In [ ]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)

with open(_run_dir / '15_validation.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['task_type', 'total', 'valid', 'invalid', 'top_reject_reason'])
    for task in sorted(task_stats):
        ts = task_stats[task]
        total = ts['total']
        valid = ts['valid']
        invalid = total - valid
        # Find top reject reason for this task type
        task_errors = Counter()
        for s in validated:
            if s.get('task_type') == task and not s['valid']:
                task_errors[s.get('validation_error', 'unknown')[:80]] += 1
        top_reason = task_errors.most_common(1)[0][0] if task_errors else ''
        w.writerow([task, total, valid, invalid, top_reason])

print(f'Saved: {_run_dir / "15_validation.csv"}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '15_validation.png'

_tasks    = sorted(task_stats.keys())
_valid    = [task_stats[t]['valid'] for t in _tasks]
_invalid  = [task_stats[t]['total'] - task_stats[t]['valid'] for t in _tasks]
_total    = [task_stats[t]['total'] for t in _tasks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: stacked bar per task type ──────────────────────────────────────────
x = range(len(_tasks))
ax1.bar(x, _valid, label='Valid', color='#2ecc71', edgecolor='white')
ax1.bar(x, _invalid, bottom=_valid, label='Invalid / no code', color='#e74c3c', edgecolor='white')
ax1.set_xticks(list(x))
ax1.set_xticklabels([t.replace('_', '\n') for t in _tasks], fontsize=8)
ax1.set_ylabel('Samples')
ax1.set_title('Validation by Task Type', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
for xi, (v, tot) in enumerate(zip(_valid, _total)):
    pct = 100 * v // max(1, tot)
    ax1.text(xi, tot + 0.5, f'{pct}%', ha='center', va='bottom', fontsize=8)
ax1.set_facecolor('#f9f9f9')

# ── Right: overall donut ──────────────────────────────────────────────────────
_n_valid   = stats['valid']
_n_invalid = stats['invalid']
_n_no_code = stats['no_code']
_n_skipped = stats['skipped']
_sizes  = [_n_valid, _n_invalid, _n_no_code, _n_skipped]
_labels = [f'Valid\n({_n_valid})', f'Invalid\n({_n_invalid})',
           f'No code\n({_n_no_code})', f'Skipped\n({_n_skipped})']
_colors = ['#2ecc71', '#e74c3c', '#f39c12', '#bdc3c7']
_non_zero = [(s, l, c) for s, l, c in zip(_sizes, _labels, _colors) if s > 0]
if _non_zero:
    s2, l2, c2 = zip(*_non_zero)
    wedges, _, autotexts = ax2.pie(
        s2, labels=l2, colors=c2, autopct='%1.0f%%',
        startangle=90, pctdistance=0.78,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for t in autotexts:
        t.set_fontsize(9); t.set_fontweight('bold')
    ax2.add_artist(plt.Circle((0, 0), 0.52, fc='white'))
    _pass_rate = 100 * _n_valid // max(1, sum(s2))
    ax2.text(0, 0, f'{_pass_rate}%\nvalid', ha='center', va='center',
             fontsize=13, fontweight='bold', color='#2c3e50')
ax2.set_title('Overall Pass Rate', fontweight='bold')

fig.suptitle(
    f'Validation — {len(validated):,} samples  ·  {len(valid_samples):,} passed to dataset',
    fontsize=13, fontweight='bold', y=1.01
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
# ── Export all ultimately dropped pairs ───────────────────────────────────────
import csv
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
broken_path = _run_dir / '16_broken.csv'

dropped = [s for s in validated if s['valid'] is False]

with open(broken_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['task_type', 'instruction', 'validation_error',
                'auto_fix_attempted', 'code_preview'])
    for s in dropped:
        task = s.get('task_type', '')
        instr = (s.get('instruction', '') or '')[:200]
        err = (s.get('validation_error', '') or '')[:200]
        attempted = 'no' if err in NOT_FIXABLE_ERRORS else 'yes'
        blocks = extract_aro_blocks(s.get('output', ''))
        preview = blocks[0][:200].replace('\n', '\\n') if blocks else ''
        w.writerow([task, instr, err, attempted, preview])

auto_fixed_count = sum(1 for s in validated if s.get('auto_fixed'))
print(f'Dropped pairs: {broken_path} ({len(dropped)} rows)')
print(f'Auto-fixed (recovered): {auto_fixed_count}')